# AMRPA + CAM Library — Local Test Notebook

Run this locally (M2 Pro or any machine) to verify the library works.
No HuggingFace downloads needed — uses mock models.

In [ ]:
import sys, os
# amrpa_lib/ must be in path
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from amrpa import AMRPAConfig, CAMConfig, AMRPAModel
from amrpa.core import AMRPACore
from amrpa.adapters.encoder import apply_amrpa_to_encoder
from amrpa.adapters.decoder import apply_amrpa_to_decoder

torch.manual_seed(42)
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')
print('AMRPA library loaded ✓')

## 1. Config

In [ ]:
enc_config = AMRPAConfig.for_encoder(d_model=768, n_heads=12)
dec_config = AMRPAConfig.for_decoder(d_model=768, n_heads=12)

print('Encoder config:')
print(f'  {enc_config}')
print(f'  causal:  {enc_config.causal}')
print(f'  use_cam: {enc_config.use_cam}')
print()
print('Decoder config:')
print(f'  {dec_config}')
print(f'  causal:  {dec_config.causal}')
print(f'  use_cam: {dec_config.use_cam}')

assert enc_config.d_k == 64 and dec_config.d_k == 64
print('\nConfig ✓')

## 2. AMRPACore — The 4 Claims

In [ ]:
core = AMRPACore(enc_config)
Q = torch.randn(2, 32, 64)
K = torch.randn(2, 32, 64)
V = torch.randn(2, 32, 64)
A = F.softmax(torch.randn(2, 32, 32), dim=-1)

# No history
bias0, m0 = core(Q, K, V, [], relative_layer_idx=1)
print(f'Layer 1 no history: bias_max={bias0.abs().max():.4f} (expect ~0)')

# With history
bias1, m1 = core(Q, K, V, [A], relative_layer_idx=2)
print(f'Layer 2 with history: bias_max={bias1.abs().max():.4f}')
print(f'  gate_impact:  {m1["gate_impact"].mean():.4f}')
print(f'  alpha_div:    {m1["alpha_diversity"].mean():.4f}')
print(f'  mem_contrib:  {m1["memory_contribution"].mean():.4f}')

# CLAIM 3: Decay
import numpy as np
steps = range(1, 11)
decay = [enc_config.gamma**k for k in steps]
plt.figure(figsize=(8,3))
plt.plot(steps, decay, 'o-', color='darkred', linewidth=2)
plt.fill_between(steps, 0, decay, alpha=0.3, color='red')
plt.title(f'CLAIM 3: Memory Decay (γ={enc_config.gamma})')
plt.xlabel('Steps back'); plt.ylabel('γ^k'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# CLAIM 4: Adaptive window
windows = [core.adaptive_window_size(l) for l in range(1, 13)]
print(f'CLAIM 4 window sizes: {windows}')

## 3. Encoder Adapter (Mock RoBERTa)

In [ ]:
class MockSA(nn.Module):
    def __init__(self):
        super().__init__()
        self.query=nn.Linear(768,768); self.key=nn.Linear(768,768); self.value=nn.Linear(768,768)
        self._in_features = 768
    @property
    def in_features(self): return self._in_features

class MockAttn(nn.Module):
    def __init__(self): super().__init__(); self.self=MockSA()

class MockBlock(nn.Module):
    def __init__(self): super().__init__(); self.attention=MockAttn()

class MockRoBERTa(nn.Module):
    def __init__(self):
        super().__init__()
        self.config=type('C',(),{'hidden_size':768,'num_attention_heads':12,'model_type':'roberta'})()
        self.encoder=type('E',(),{'layer':nn.ModuleList([MockBlock() for _ in range(12)])})()
        self.embeddings=nn.Embedding(100,768)

config_enc = AMRPAConfig.for_encoder(d_model=768, n_heads=12)
config_enc.freeze_embeddings = False

roberta = MockRoBERTa()
model_e, state_e = apply_amrpa_to_encoder(roberta, config_enc)
print(f'Encoder patched: {len(state_e.layers)} AMRPA layers ✓')

state_e.reset()
layer_e = state_e.layers[1]

print('\nRunning 4 encoder steps:')
for step in range(4):
    out = layer_e(torch.randn(2, 32, 768), attention_mask=torch.zeros(2,1,1,32))
    metrics = state_e.get_metrics()
    gate = metrics.get('gate_impact', torch.zeros(1)).mean().item()
    print(f'  Step {step}: out={out[0].shape}, gate={gate:.4f}')

print('Encoder ✓')

## 4. Decoder Adapter (Mock GPT2) — AMRPA + CAM

In [ ]:
class MockGPT2Attn(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed_dim=768; self.num_heads=12
        self.c_attn=nn.Linear(768,768*3); self.c_proj=nn.Linear(768,768)
        self.resid_dropout=nn.Dropout(0.0)

class MockGPT2Block(nn.Module):
    def __init__(self): super().__init__(); self.attn=MockGPT2Attn()

class MockGPT2(nn.Module):
    def __init__(self):
        super().__init__()
        self.config=type('C',(),{'model_type':'gpt2','n_embd':768,'n_head':12,'hidden_size':768,'num_attention_heads':12})()
        self.h=nn.ModuleList([MockGPT2Block() for _ in range(12)])

config_dec = AMRPAConfig.for_decoder(d_model=768, n_heads=12)
gpt2 = MockGPT2()
model_d, state_d = apply_amrpa_to_decoder(gpt2, config_dec)

state_d.reset()
layer_d = state_d.layers[1]

print('Simulating decoder generation (growing sequence):')
steps_data = []
for step in range(8):
    seq_t = 8 + step
    out   = layer_d(torch.randn(2, seq_t, 768))
    m     = state_d.get_metrics()
    gate  = m.get('gate_impact', torch.zeros(1)).mean().item()
    steps_data.append({'step':step,'seq':seq_t,'gate':gate})
    print(f'  Step {step} seq={seq_t}: out={out[0].shape}, gate={gate:.4f}')

mem_summary = state_d.cam_bank.summary()
print(f'\nCAM memory: {mem_summary["total_memory_mb"]:.2f} MB')
print(f'Stored per layer: {mem_summary["stored_per_layer"]}')
print('Decoder ✓')

## 5. Metrics Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
steps_x = [d['step'] for d in steps_data]
gates   = [d['gate']  for d in steps_data]

axes[0].plot(steps_x, gates, 'o-', color='orange', linewidth=2)
axes[0].fill_between(steps_x, 0, gates, alpha=0.3, color='orange')
axes[0].set_title('Decoder Gate Impact per Step')
axes[0].set_xlabel('Generation step'); axes[0].set_ylabel('Gate')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Step 0: no memory yet', xy=(0, gates[0]),
                 xytext=(1, max(gates)*0.5),
                 arrowprops=dict(arrowstyle='->', color='gray'))

decay_vals = [config_dec.gamma**k for k in range(1,9)]
axes[1].plot(range(1,9), decay_vals, 'o-', color='darkred', linewidth=2)
axes[1].fill_between(range(1,9), 0, decay_vals, alpha=0.3, color='red')
axes[1].set_title(f'CLAIM 3: Decay (γ={config_dec.gamma})')
axes[1].set_xlabel('Steps back'); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Summary

In [ ]:
print('='*55)
print('AMRPA + CAM LIBRARY — LOCAL TEST SUMMARY')
print('='*55)
checks = [
    ('AMRPAConfig encoder',          not enc_config.causal and not enc_config.use_cam),
    ('AMRPAConfig decoder',          dec_config.causal and dec_config.use_cam),
    ('AMRPACore no history = zero',  bias0.abs().max().item() < 1e-6),
    ('AMRPACore with history fires', bias1.abs().max().item() > 0),
    ('Encoder 4 layers patched',     len(state_e.layers) == 4),
    ('Decoder 4 layers patched',     len(state_d.layers) == 4),
    ('Decoder sequence grows',       True),
    ('CAM memory bounded',           max(mem_summary['stored_per_layer']) <= config_dec.cam.window_size),
]
all_pass = True
for name, result in checks:
    icon = '✅' if result else '❌'
    print(f'  {icon}  {name}')
    if not result: all_pass = False
print('='*55)
if all_pass:
    print('ALL PASSED — library ready for Kaggle training.')
else:
    print('SOME FAILED — review cells above.')
print('='*55)